# Bases de données — le cours à lire avant les exercices

Ce cours se lit **avant** d'attaquer les exercices du polycopié. Il part de zéro : il suffit de savoir ce qu'est un tableau de données (un fichier CSV, vu en Première).

Il suit le **programme officiel** de Terminale NSI (thème « Bases de données », publié par le ministère de l'Éducation nationale), dont voici les capacités attendues :

| Notion du programme | Ce qu'il faut savoir faire | Dans ce cours |
|:---|:---|:---|
| Modèle relationnel : relation, attribut, domaine, clef primaire, clef étrangère, schéma relationnel | identifier ces notions | parties 1, 9 et 10 |
| Base de données relationnelle | distinguer la **structure** d'une base de son **contenu** ; repérer des anomalies (redondances) dans un schéma | parties 1 et 11 |
| Système de gestion de bases de données (SGBD) | connaître ses services : persistance, accès concurrents, efficacité, sécurité | partie 2 |
| Langage SQL | construire des requêtes avec `SELECT`, `FROM`, `WHERE`, `JOIN` ; `INSERT`, `UPDATE`, `DELETE` ; on peut utiliser `DISTINCT`, `ORDER BY` et les fonctions d'agrégation, **sans** `GROUP BY` ni `HAVING` | parties 4 à 8, 10 et 12 |

**Comment lire ce cours :** chaque notion est expliquée, puis montrée sur un **exemple tapé dans la console H2** (le logiciel utilisé en classe). Le résultat affiché sous chaque exemple est celui qu'a réellement donné H2. Le mieux est de retaper les exemples soi-même.

**Plan :**
1. Le vocabulaire : relation, attribut, enregistrement, domaine
2. Le SGBD et la console H2
3. Les types (domaines de valeurs)
4. Créer une table : `CREATE TABLE`
5. Remplir une table : `INSERT INTO`, et `RUNSCRIPT`
6. Les outils Python de l'exercice 1 : lire un CSV, écrire un fichier
7. Interroger une table : `SELECT … FROM … WHERE …`
8. Compter, moyenne, minimum, maximum
9. Clé primaire
10. Plusieurs tables : clé étrangère et jointure `JOIN`
11. Pourquoi plusieurs tables ? (les anomalies)
12. Modifier et supprimer : `UPDATE`, `DELETE`
13. La méthode pour écrire une requête, et la fiche récapitulative

---
# 1. Le vocabulaire

Une **base de données relationnelle** range les données dans des **tableaux**. Exemple : un tableau des pays.

| id | code | name | continent |
|:---|:---|:---|:---|
| 1 | FR | France | EU |
| 2 | ES | Spain | EU |
| 3 | BR | Brazil | SA |

Les mots à connaître (le cours du polycopié emploie les deux colonnes) :

| Mot du cours | Mot de tous les jours | Dans l'exemple |
|:---|:---|:---|
| **relation** | table, tableau | la table `Countries` |
| **attribut** | colonne | `id`, `code`, `name`, `continent` |
| **enregistrement** (ou n-uplet) | ligne | `1, FR, France, EU` |
| **domaine** d'un attribut | les valeurs autorisées, son **type** | `id` : un entier ; `name` : du texte |

- Chaque **relation** décrit une sorte d'objets du monde réel (des pays, des aéroports, des élèves…). Chaque **ligne** est un objet ; chaque **colonne** est une caractéristique de ces objets.
- **Structure et contenu** : la **structure** d'une table, c'est la liste de ses colonnes et de leurs types (elle change rarement) ; le **contenu**, ce sont les lignes (elles changent tout le temps). On décrit la structure par un **schéma** :

  Countries(id : INT, code : CHAR(2), name : VARCHAR, continent : VARCHAR)

- **Pourquoi pas un simple fichier CSV ?** Un CSV convient pour quelques centaines de lignes lues par un seul programme. Une base de données gère des millions de lignes, utilisées **en même temps** par beaucoup d'utilisateurs, en **vérifiant** que les données restent cohérentes. C'est le rôle du SGBD.

---
# 2. Le SGBD et la console H2

Un **SGBD** (système de gestion de bases de données) est le logiciel qui stocke la base et répond aux demandes. En classe, on utilise le SGBD **H2**. D'autres exemples : SQLite, MySQL, PostgreSQL, Oracle.

**Les services rendus par un SGBD** (à savoir citer) :
- **persistance** : les données sont enregistrées sur le disque, elles restent quand on éteint l'ordinateur ;
- **gestion des accès concurrents** : plusieurs utilisateurs peuvent lire et modifier la base **en même temps** sans tout mélanger (deux personnes ne peuvent pas réserver le même siège) ;
- **efficacité** : il trouve très vite une ligne parmi des millions ;
- **sécurisation des accès** : chaque utilisateur a un mot de passe et des droits (lire seulement, ou modifier) ;
- en plus, il **vérifie la cohérence** des données (les *contraintes d'intégrité*, parties 3, 9 et 10).

**On parle au SGBD avec le langage SQL.** Chaque demande s'appelle une **requête** (ou instruction) et se termine par un **point-virgule** `;`. Les majuscules aux mots-clés (`SELECT`, `FROM`…) sont une habitude de lecture, pas une obligation. Les retours à la ligne et les espaces n'ont pas d'importance.

**Utiliser la console H2 :**
1. lancer « H2 Console » : une page s'ouvre dans le navigateur ;
2. se connecter : JDBC URL `jdbc:h2:~/nsi` (le fichier de la base, dans le dossier personnel), utilisateur `sa`, mot de passe vide, bouton **Connecter** ;
3. taper la requête dans la grande zone de texte, puis cliquer sur **Exécuter** (ou Ctrl + Entrée) ;
4. le résultat s'affiche en dessous, sous forme de tableau.

On peut taper plusieurs requêtes d'un coup (chacune finie par `;`) : H2 les exécute dans l'ordre.

> Pour repartir d'une base vide : `DROP ALL OBJECTS;` efface toutes les tables de la base connectée.

---
# 3. Les types (domaines de valeurs)

Chaque colonne a un **type** : le SGBD **refuse** une valeur qui n'est pas du bon type. C'est la **contrainte de domaine**.

| Type | Contenu | Exemple de valeur en SQL |
|:---|:---|:---|
| `INT` | nombre entier | `302687` |
| `REAL` (ou `FLOAT`) | nombre à virgule (avec un **point**) | `48.72` |
| `CHAR(n)` | texte d'**au plus** n caractères | `'FR'` pour `CHAR(2)` |
| `VARCHAR` | texte de longueur quelconque | `'France'` |
| `DATE` | une date | `'2026-11-14'` (année-mois-jour) |
| `TIMESTAMP` | une date et une heure | `'2026-11-14 08:30:00'` |
| `BOOLEAN` | vrai ou faux | `TRUE`, `FALSE` |

⚠️ **Le texte s'écrit entre apostrophes** `'...'` (pas entre guillemets). Tout ce qui est entre les apostrophes compte, **espaces compris** : `' FR '` fait 4 caractères. Les nombres s'écrivent **sans** apostrophes.

---
# 4. Créer une table : `CREATE TABLE`

On décrit la **structure** : le nom de la table, puis chaque colonne avec son type, séparées par des virgules (pas de virgule après la dernière).

*Dans la console H2 :*

```sql
CREATE TABLE Countries (
    id INT,
    code CHAR(2),
    name VARCHAR,
    continent VARCHAR );
```

*Résultat dans la console H2 :*

```text
(Update count: 0)
```

- `Update count: 0` : la table est créée, elle est **vide** (aucune ligne modifiée).
- Si on l'exécute une deuxième fois, H2 répond que la table existe déjà. Pour la supprimer (structure **et** contenu) : `DROP TABLE Countries;`.
- C'est exactement la question 1 de l'exercice 1 du polycopié.

---
# 5. Remplir une table : `INSERT INTO`

Une requête `INSERT INTO` ajoute **une ligne**. On donne les colonnes, puis les valeurs **dans le même ordre**.

*Dans la console H2 :*

```sql
INSERT INTO Countries (id, code, name, continent) VALUES (1, 'FR', 'France', 'EU');
INSERT INTO Countries (id, code, name, continent) VALUES (2, 'ES', 'Spain', 'EU');
INSERT INTO Countries (id, code, name, continent) VALUES (3, 'BR', 'Brazil', 'SA');
SELECT * FROM Countries;
```

*Résultat dans la console H2 :*

```text
INSERT INTO Countries (id, code, name, continent) VALUES (1, 'FR', 'France', 'EU');
(Update count: 1)

INSERT INTO Countries (id, code, name, continent) VALUES (2, 'ES', 'Spain', 'EU');
(Update count: 1)

INSERT INTO Countries (id, code, name, continent) VALUES (3, 'BR', 'Brazil', 'SA');
(Update count: 1)

SELECT * FROM Countries;
ID | CODE | NAME   | CONTINENT
1  | FR   | France | EU
2  | ES   | Spain  | EU
3  | BR   | Brazil | SA
(3 rows)
```

- `Update count: 1` : une ligne ajoutée. `SELECT * FROM Countries;` affiche toute la table (partie 7).
- Si on donne une valeur pour **toutes** les colonnes, dans l'ordre de la table, on peut omettre la liste des colonnes : `INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');`.

**Le SGBD vérifie les types.** Une chaîne de 4 caractères dans une colonne `CHAR(2)` est refusée :

*Dans la console H2 :*

```sql
INSERT INTO Countries VALUES (4, ' IM ', 'Isle of Man', 'EU');
```

*Résultat dans la console H2 :*

```text
Erreur : Valeur trop longue pour la colonne "CODE CHARACTER(2)": "' IM ' (4)"
Value too long for column "CODE CHARACTER(2)": "' IM ' (4)"; SQL statement:
INSERT INTO Countries VALUES (4, ' IM ', 'Isle of Man', 'EU') [22001-252]
```

⚠️ C'est **le piège du copier-coller depuis le PDF** du polycopié : il ajoute des espaces autour des apostrophes. Il faut taper `'IM'`, collé aux apostrophes.

**Une apostrophe dans un texte** (*Côte d'Ivoire*) : comme l'apostrophe termine le texte, on la **double** : `'Côte d''Ivoire'`.

*Dans la console H2 :*

```sql
INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');
INSERT INTO Countries VALUES (5, 'CI', 'Côte d''Ivoire', 'AF');
INSERT INTO Countries VALUES (6, 'JP', 'Japan', 'AS');
SELECT * FROM Countries;
```

*Résultat dans la console H2 :*

```text
INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');
(Update count: 1)

INSERT INTO Countries VALUES (5, 'CI', 'Côte d''Ivoire', 'AF');
(Update count: 1)

INSERT INTO Countries VALUES (6, 'JP', 'Japan', 'AS');
(Update count: 1)

SELECT * FROM Countries;
ID | CODE | NAME          | CONTINENT
1  | FR   | France        | EU
2  | ES   | Spain         | EU
3  | BR   | Brazil        | SA
4  | IM   | Isle of Man   | EU
5  | CI   | Côte d'Ivoire | AF
6  | JP   | Japan         | AS
(6 rows)
```

### Exécuter un fichier : `RUNSCRIPT`

Un fichier `.sql` est un simple **fichier texte** qui contient des requêtes, l'une après l'autre. Au lieu de les recopier, on demande à H2 de toutes les exécuter :

```sql
RUNSCRIPT FROM 'C:/Users/Iheb/Downloads/countries.sql';
```

(entre apostrophes, le chemin complet du fichier, avec des `/` même sous Windows ; sur Mac : `'/Users/iheb/Downloads/countries.sql'`). C'est ainsi qu'on charge les données des exercices 2 (aéroports) et 3 (Vélib').

---
# 6. Les outils Python de l'exercice 1

La question 2 (b) de l'exercice 1 demande un programme Python qui **fabrique** le fichier `countries.sql` à partir du fichier `countries.csv`. Il faut trois outils.

### Outil 1 : écrire dans un fichier texte

```python
fichier = open("essai.txt", "w", encoding="utf-8")   # "w" = write : on crée le fichier (ou on l'efface)
fichier.write("première ligne\n")                      # "\n" = passer à la ligne
fichier.write("deuxième ligne\n")
fichier.close()                                         # obligatoire : c'est là que tout est enregistré
```
`write` écrit **exactement** la chaîne donnée : sans `"\n"`, tout serait sur la même ligne.

### Outil 2 : lire un fichier CSV avec le module `csv`

**Étape 1 — Ce qu'est un fichier CSV.** C'est un simple **fichier texte** qui contient un tableau. Chaque ligne du fichier est une ligne du tableau, et les valeurs sont séparées par des **virgules**. La première ligne donne les **noms des colonnes**.

Exemple : un petit fichier `eleves.csv` de 3 lignes.

```text
nom,prenom,classe
Martin,Léa,T3
Durand,Paul,T1
```
Il représente ce tableau :

| nom | prenom | classe |
|:---|:---|:---|
| Martin | Léa | T3 |
| Durand | Paul | T1 |

**Étape 2 — Le problème.** Si on lit ce fichier normalement, Python donne chaque ligne comme **un seul bloc de texte** :
```python
fichier = open("eleves.csv", encoding="utf-8")
for ligne in fichier:
    print(repr(ligne))
fichier.close()
```
```text
'nom,prenom,classe
'
'Martin,Léa,T3
'
'Durand,Paul,T1
'
```
Pour avoir le prénom de Léa, il faudrait découper le texte aux virgules soi-même. C'est pénible : c'est le module `csv` qui fait ce travail.

**Étape 3 — La solution : `csv.DictReader`.** Il lit le fichier **ligne par ligne** et, pour chaque ligne du tableau, il donne un **dictionnaire** :
- les **clés** sont les noms des colonnes (pris dans la première ligne du fichier) ;
- les **valeurs** sont le contenu de la ligne.

```python
import csv

fichier = open("eleves.csv", encoding="utf-8")
for ligne in csv.DictReader(fichier):
    print(ligne)
fichier.close()
```
```text
{'nom': 'Martin', 'prenom': 'Léa', 'classe': 'T3'}
{'nom': 'Durand', 'prenom': 'Paul', 'classe': 'T1'}
```
La boucle fait **2 tours** (un par élève). La ligne d'en-tête ne donne pas de tour : elle sert seulement à fabriquer les clés.

> **Image à retenir :** `DictReader` est comme quelqu'un qui lit le tableau à ta place et te tend, à chaque tour, une **fiche** : « nom → Martin, prénom → Léa, classe → T3 ».

**Étape 4 — Utiliser la fiche.** Comme `ligne` est un dictionnaire, on récupère une valeur avec le **nom de la colonne** entre crochets :
```python
fichier = open("eleves.csv", encoding="utf-8")
for ligne in csv.DictReader(fichier):
    print(ligne["prenom"], "est en", ligne["classe"])
fichier.close()
```
```text
Léa est en T3
Paul est en T1
```

| tour de boucle | `ligne` | `ligne["prenom"]` | `ligne["classe"]` |
|:---|:---|:---|:---|
| 1 | `{'nom': 'Martin', 'prenom': 'Léa', 'classe': 'T3'}` | `'Léa'` | `'T3'` |
| 2 | `{'nom': 'Durand', 'prenom': 'Paul', 'classe': 'T1'}` | `'Paul'` | `'T1'` |

**Étape 5 — Avec le vrai fichier de l'exercice 1.** C'est exactement pareil avec `countries.csv`, qui a les colonnes `id`, `code`, `name`, `continent` :
```text
"id","code","name","continent"
302672,"AD","Andorra","EU"
```
Au premier tour, `DictReader` donne :
```text
{'id': '302672', 'code': 'AD', 'name': 'Andorra', 'continent': 'EU'}
```
- les guillemets du fichier ont disparu : `csv` les enlève tout seul ;
- ⚠️ **toutes les valeurs sont des chaînes**, même l'identifiant : `'302672'` et pas `302672`. Ici, c'est pratique : on peut directement les coller avec `+` (outil 3).

### Outil 3 : fabriquer une requête SQL avec `+`

On colle des morceaux de texte avec `+` :
```python
ligne = "INSERT INTO Countries VALUES (" + pays["id"] + ", '" + pays["code"] + "', '" + pays["name"] + "', '" + pays["continent"] + "');"
```
Pour Andorre, `ligne` vaut : `INSERT INTO Countries VALUES (302672, 'AD', 'Andorra', 'EU');`

Regarder les guillemets de près : la chaîne Python `", '"` contient une virgule, une espace et **une apostrophe** — c'est cette apostrophe qui ouvre le texte SQL `'AD'`.

⚠️ **Le piège de l'apostrophe** : pour *Côte d'Ivoire*, on obtiendrait `'Côte d'Ivoire'`, refusé par H2. Il faut doubler l'apostrophe (partie 5) avant de construire la ligne, avec une petite fonction qui parcourt le texte caractère par caractère :
```python
def echappe(texte):
    resultat = ""
    for caractere in texte:
        if caractere == "'":
            resultat = resultat + "''"
        else:
            resultat = resultat + caractere
    return resultat
```

**Avec ces trois outils**, le programme de l'exercice 1 : ouvrir le CSV en lecture et `countries.sql` en écriture ; pour chaque pays, construire la ligne `INSERT` (avec `echappe` sur le nom) et l'écrire suivie de `"\n"` ; fermer les deux fichiers.

---
# 7. Interroger une table : `SELECT … FROM … WHERE …`

Jusqu'ici, on a **rangé** des données. Maintenant, on va **poser des questions** à la base : « quels aéroports sont à Paris ? », « combien y a-t-il d'aéroports en France ? ». Une question posée en SQL s'appelle une **requête d'interrogation**. Elle commence toujours par `SELECT`.

**L'idée la plus importante de cette partie : une requête `SELECT` ne modifie rien.** Elle **lit** une table et **fabrique un nouveau tableau**, le résultat, que H2 affiche. La table elle-même reste intacte.

### La table qui va servir d'exemple

Pour s'entraîner, on crée une table `Airports` de 7 aéroports. Avant chaque requête, on pourra regarder ce tableau et **deviner le résultat**, puis vérifier avec H2.

*On crée et on remplit la table `Airports` (rien de nouveau : parties 4 et 5) :*

```sql
CREATE TABLE Airports (
    code CHAR(4), name VARCHAR, type VARCHAR,
    latitude REAL, altitude INT, country CHAR(2), city VARCHAR );
INSERT INTO Airports VALUES ('LFPG', 'Charles de Gaulle Airport', 'large_airport', 49.01, 392, 'FR', 'Paris');
INSERT INTO Airports VALUES ('LFPO', 'Paris-Orly Airport', 'large_airport', 48.72, 291, 'FR', 'Paris');
INSERT INTO Airports VALUES ('LFBO', 'Toulouse-Blagnac Airport', 'large_airport', 43.63, 499, 'FR', 'Toulouse');
INSERT INTO Airports VALUES ('LFPI', 'Paris Issy Heliport', 'heliport', 48.83, 112, 'FR', 'Paris');
INSERT INTO Airports VALUES ('LEMD', 'Madrid-Barajas Airport', 'large_airport', 40.47, 1998, 'ES', 'Madrid');
INSERT INTO Airports VALUES ('SBGR', 'Guarulhos Airport', 'large_airport', -23.44, 2459, 'BR', 'Sao Paulo');
INSERT INTO Airports VALUES ('RJTT', 'Tokyo Haneda Airport', 'large_airport', 35.55, 35, 'JP', 'Tokyo');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Airports ( code CHAR(4), name VARCHAR, type VARCHAR, latitude REAL, altitude INT, country CHAR(2), city VARCHAR );
(Update count: 0)

INSERT INTO Airports VALUES ('LFPG', 'Charles de Gaulle Airport', 'large_airport', 49.01, 392, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES ('LFPO', 'Paris-Orly Airport', 'large_airport', 48.72, 291, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES ('LFBO', 'Toulouse-Blagnac Airport', 'large_airport', 43.63, 499, 'FR', 'Toulouse');
(Update count: 1)

INSERT INTO Airports VALUES ('LFPI', 'Paris Issy Heliport', 'heliport', 48.83, 112, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES ('LEMD', 'Madrid-Barajas Airport', 'large_airport', 40.47, 1998, 'ES', 'Madrid');
(Update count: 1)

INSERT INTO Airports VALUES ('SBGR', 'Guarulhos Airport', 'large_airport', -23.44, 2459, 'BR', 'Sao Paulo');
(Update count: 1)

INSERT INTO Airports VALUES ('RJTT', 'Tokyo Haneda Airport', 'large_airport', 35.55, 35, 'JP', 'Tokyo');
(Update count: 1)
```

Voici donc la table `Airports` (à garder sous les yeux pour toute la suite) :

| code | name | type | latitude | altitude | country | city |
|:---|:---|:---|:---|:---|:---|:---|
| LFPG | Charles de Gaulle Airport | large_airport | 49.01 | 392 | FR | Paris |
| LFPO | Paris-Orly Airport | large_airport | 48.72 | 291 | FR | Paris |
| LFBO | Toulouse-Blagnac Airport | large_airport | 43.63 | 499 | FR | Toulouse |
| LFPI | Paris Issy Heliport | heliport | 48.83 | 112 | FR | Paris |
| LEMD | Madrid-Barajas Airport | large_airport | 40.47 | 1998 | ES | Madrid |
| SBGR | Guarulhos Airport | large_airport | -23.44 | 2459 | BR | Sao Paulo |
| RJTT | Tokyo Haneda Airport | large_airport | 35.55 | 35 | JP | Tokyo |

(`latitude` : positive au nord de l'équateur, négative au sud ; `altitude` en pieds ; `country` : le code du pays.)

---
## 7.1 Choisir les colonnes : `SELECT … FROM …`

**Le besoin :** « Je veux la liste des noms d'aéroports et de leurs villes. » On ne veut pas tout le tableau, seulement **deux colonnes**.

**La syntaxe :**
```sql
SELECT name, city FROM Airports;
```
Elle se lit comme une phrase :

| morceau | sens |
|:---|:---|
| `SELECT name, city` | « montre-moi les colonnes `name` et `city` » (séparées par une virgule) |
| `FROM Airports` | « en les prenant dans la table `Airports` » |
| `;` | fin de la requête |

**Ce que fait H2 :** il prend la table, **garde seulement les colonnes demandées**, et garde **toutes les lignes**. Devine le résultat, puis vérifie :

*Dans la console H2 :*

```sql
SELECT name, city FROM Airports;
```

*Résultat dans la console H2 :*

```text
NAME                      | CITY
Charles de Gaulle Airport | Paris
Paris-Orly Airport        | Paris
Toulouse-Blagnac Airport  | Toulouse
Paris Issy Heliport       | Paris
Madrid-Barajas Airport    | Madrid
Guarulhos Airport         | Sao Paulo
Tokyo Haneda Airport      | Tokyo
(7 rows)
```

- Il y a bien 7 lignes, mais 2 colonnes seulement. H2 écrit les noms de colonnes en majuscules.
- `SELECT * FROM Airports;` : l'étoile `*` veut dire « **toutes** les colonnes ». C'est la requête à taper en premier pour **découvrir** une table.

✏️ **Vérifie que tu as compris :** quelle requête affiche seulement les codes et les altitudes ? *(Réponse : `SELECT code, altitude FROM Airports;`)*

---
## 7.2 Choisir les lignes : `WHERE`

**Le besoin :** « Je veux les noms des aéroports **de Paris**. » Cette fois, on ne veut pas toutes les lignes : seulement celles qui vérifient une **condition**.

**L'idée :** `WHERE` est un **filtre**. H2 regarde les lignes **une par une**, teste la condition sur chacune, et ne garde que celles où elle est **vraie**. Pour la condition `city = 'Paris'` :

| code | city | `city = 'Paris'` ? | gardée ? |
|:---|:---|:---|:---|
| LFPG | Paris | vrai | ✔ |
| LFPO | Paris | vrai | ✔ |
| LFBO | Toulouse | faux | ✘ |
| LFPI | Paris | vrai | ✔ |
| LEMD | Madrid | faux | ✘ |
| SBGR | Sao Paulo | faux | ✘ |
| RJTT | Tokyo | faux | ✘ |

Puis `SELECT` choisit les colonnes à afficher, **parmi les lignes gardées**.

**La syntaxe :** on ajoute `WHERE condition` **après** le `FROM`.
```sql
SELECT name FROM Airports WHERE city = 'Paris';
```

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE city = 'Paris';
```

*Résultat dans la console H2 :*

```text
NAME
Charles de Gaulle Airport
Paris-Orly Airport
Paris Issy Heliport
(3 rows)
```

**Les comparaisons possibles :**

| en SQL | sens | exemple |
|:---|:---|:---|
| `=` | égal (un seul `=`, pas `==` comme en Python) | `country = 'FR'` |
| `<>` | différent | `type <> 'heliport'` |
| `<`, `>`, `<=`, `>=` | plus petit, plus grand… | `altitude > 1000` |

Les textes sont entre apostrophes (`'Paris'`), les nombres sans (`1000`).

**Plusieurs conditions :** on les relie avec `AND` (les deux doivent être vraies), `OR` (au moins une), comme `and` et `or` en Python.

*« Les grands aéroports de Paris »* : il faut être à Paris **et** être un grand aéroport. L'héliport de Paris est écarté.

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE city = 'Paris' AND type = 'large_airport';
```

*Résultat dans la console H2 :*

```text
NAME
Charles de Gaulle Airport
Paris-Orly Airport
(2 rows)
```

*« Les aéroports très hauts (plus de 1000 pieds) ou au Japon »* : il suffit d'une des deux conditions.

*Dans la console H2 :*

```sql
SELECT code, altitude FROM Airports WHERE altitude > 1000 OR country = 'JP';
```

*Résultat dans la console H2 :*

```text
CODE | ALTITUDE
LEMD | 1998
SBGR | 2459
RJTT | 35
(3 rows)
```

**L'ordre dans lequel H2 travaille** (différent de l'ordre d'écriture !) :
1. `FROM` : quelle table ?
2. `WHERE` : quelles lignes garder ?
3. `SELECT` : quelles colonnes afficher ?

C'est aussi l'ordre dans lequel il faut **lire** une requête pour la comprendre.

✏️ **Vérifie que tu as compris :** sans H2, quel est le résultat de `SELECT city FROM Airports WHERE country = 'FR' AND altitude < 300;` ? *(Réponse : deux lignes, Paris (Orly, 291) et Paris (Issy, 112).)*

---
## 7.3 Chercher un mot dans un texte : `LIKE`

**Le besoin :** « Les aéroports dont le nom **contient** le mot Paris. » Avec `=`, il faudrait connaître le nom **exact**. `LIKE` permet de chercher un **morceau** de texte.

**L'idée :** on écrit un **motif**, où le symbole `%` veut dire « n'importe quoi » (n'importe quels caractères, ou rien du tout).

| motif | sens | `'Paris-Orly Airport'` correspond ? |
|:---|:---|:---|
| `'Paris%'` | commence par Paris | ✔ |
| `'%Airport'` | finit par Airport | ✔ |
| `'%Orly%'` | contient Orly | ✔ |
| `'%Tokyo%'` | contient Tokyo | ✘ |

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE name LIKE '%Paris%';
```

*Résultat dans la console H2 :*

```text
NAME
Paris-Orly Airport
Paris Issy Heliport
(2 rows)
```

- « Charles de Gaulle Airport » n'est **pas** dans le résultat : son nom ne contient pas le mot Paris (même si l'aéroport est à Paris !). On cherche dans le **texte** de la colonne `name`, pas dans le sens.
- ⚠️ Dans H2, `LIKE` **distingue majuscules et minuscules** : `'%paris%'` ne trouverait rien ici.

---
## 7.4 Trier les résultats : `ORDER BY`

**Le besoin :** « Je veux les aéroports français, du plus haut au plus bas. » On sait déjà garder les aéroports français (avec `WHERE`) et afficher leur nom et leur altitude (avec `SELECT`). Ce qui est nouveau, c'est l'**ordre** des lignes.

**Étape 1 — Ce que donne la requête sans tri.** H2 affiche les lignes dans l'ordre où elles ont été enregistrées dans la table :

*Dans la console H2 :*

```sql
SELECT name, altitude FROM Airports WHERE country = 'FR';
```

*Résultat dans la console H2 :*

```text
NAME                      | ALTITUDE
Charles de Gaulle Airport | 392
Paris-Orly Airport        | 291
Toulouse-Blagnac Airport  | 499
Paris Issy Heliport       | 112
(4 rows)
```

Les altitudes arrivent dans le désordre : 392, 291, 499, 112. On ne voit pas tout de suite quel aéroport est le plus haut.

**Étape 2 — Trier.** **Trier**, c'est ranger les lignes du résultat selon les valeurs d'une colonne. En SQL, on ajoute **à la fin** de la requête `ORDER BY` (« trier selon ») suivi du nom de la colonne qui sert au tri. Ici, on trie selon l'altitude :

*Dans la console H2 :*

```sql
SELECT name, altitude FROM Airports WHERE country = 'FR' ORDER BY altitude;
```

*Résultat dans la console H2 :*

```text
NAME                      | ALTITUDE
Paris Issy Heliport       | 112
Paris-Orly Airport        | 291
Charles de Gaulle Airport | 392
Toulouse-Blagnac Airport  | 499
(4 rows)
```

Les lignes sont maintenant triées dans l'**ordre croissant** : du plus petit au plus grand (112, 291, 392, 499). C'est l'ordre choisi par défaut, quand on n'écrit rien après le nom de la colonne. Pour une colonne de texte, l'ordre croissant est l'ordre alphabétique (de A à Z).

**Étape 3 — Choisir le sens du tri.** On voulait l'inverse : du plus haut au plus bas, c'est-à-dire l'**ordre décroissant** (du plus grand au plus petit). On ajoute le mot `DESC` (de l'anglais *descending*, « décroissant ») après le nom de la colonne :

*Dans la console H2 :*

```sql
SELECT name, altitude FROM Airports WHERE country = 'FR' ORDER BY altitude DESC;
```

*Résultat dans la console H2 :*

```text
NAME                      | ALTITUDE
Toulouse-Blagnac Airport  | 499
Charles de Gaulle Airport | 392
Paris-Orly Airport        | 291
Paris Issy Heliport       | 112
(4 rows)
```

On obtient bien les aéroports français du plus haut (Toulouse, 499) au plus bas (l'héliport d'Issy, 112).

**À retenir :**

| on écrit à la fin de la requête | tri obtenu |
|:---|:---|
| `ORDER BY colonne` | croissant : du plus petit au plus grand, de A à Z |
| `ORDER BY colonne DESC` | décroissant : du plus grand au plus petit, de Z à A |

`ORDER BY` ne change pas **quelles** lignes sont affichées (c'est le rôle de `WHERE`) : il change seulement **l'ordre** dans lequel elles sont affichées.

---
## 7.5 Enlever les doublons : `DISTINCT`

**Le besoin :** « Dans quelles villes françaises y a-t-il un aéroport ? »

**Étape 1 — La requête naturelle.** On veut la colonne `city`, pour les aéroports français :

*Dans la console H2 :*

```sql
SELECT city FROM Airports WHERE country = 'FR';
```

*Résultat dans la console H2 :*

```text
CITY
Paris
Paris
Toulouse
Paris
(4 rows)
```

**Étape 2 — Le problème.** Paris apparaît **trois fois**. Ce n'est pas une erreur : il y a trois aéroports français à Paris (Charles de Gaulle, Orly et l'héliport d'Issy), et H2 affiche une ligne par aéroport gardé. Une valeur qui apparaît plusieurs fois dans un résultat s'appelle un **doublon**. Or la question demande les **villes**, et chaque ville ne devrait apparaître qu'une fois.

**Étape 3 — `DISTINCT`.** On écrit le mot `DISTINCT` juste après `SELECT`. H2 affiche alors chaque valeur **une seule fois** (*distinct* veut dire « différent » : on ne garde que des valeurs différentes) :

*Dans la console H2 :*

```sql
SELECT DISTINCT city FROM Airports WHERE country = 'FR';
```

*Résultat dans la console H2 :*

```text
CITY
Paris
Toulouse
(2 rows)
```

Il reste deux lignes : Paris et Toulouse. C'est la réponse à la question.

**À retenir :** quand une question demande « les différents… », « quelles villes… », « quels types… », on se demande si une même valeur peut revenir plusieurs fois dans le résultat. Si oui, on écrit `SELECT DISTINCT`.

**L'ordre d'écriture complet** d'une requête d'interrogation (on peut sauter les parties inutiles, mais pas changer l'ordre) :
```sql
SELECT [DISTINCT] colonnes
FROM table
WHERE condition
ORDER BY colonne [DESC];
```

---
# 8. Résumer une colonne en un seul nombre : les fonctions d'agrégation

**Le besoin :** parfois on ne veut pas une **liste**, mais **un seul nombre** qui résume : « **combien** y a-t-il d'aéroports en France ? », « quelle est l'altitude **moyenne** ? », « quelle est la **plus grande** altitude ? ».

**L'idée :** une fonction d'agrégation prend **toute une colonne** (toutes les lignes gardées par le `WHERE`) et la **résume en une seule valeur**.

| fonction | ce qu'elle calcule | sur la colonne altitude des 7 aéroports |
|:---|:---|:---|
| `COUNT(*)` | le **nombre** de lignes | 7 |
| `SUM(altitude)` | la **somme** | 5786 |
| `AVG(altitude)` | la **moyenne** (average) | 5786 ÷ 7 ≈ 826,6 |
| `MIN(altitude)` | la **plus petite** valeur | 35 |
| `MAX(altitude)` | la **plus grande** valeur | 2459 |

**Avec un `WHERE`**, le filtre agit **d'abord**, puis la fonction résume les lignes qui restent. *« Combien d'aéroports en France ? »* : on garde les 4 lignes françaises, puis `COUNT(*)` les compte.

*Dans la console H2 :*

```sql
SELECT COUNT(*) FROM Airports WHERE country = 'FR';
SELECT AVG(altitude) FROM Airports;
SELECT MIN(latitude), MAX(altitude) FROM Airports;
```

*Résultat dans la console H2 :*

```text
SELECT COUNT(*) FROM Airports WHERE country = 'FR';
COUNT(*)
4
(1 row)

SELECT AVG(altitude) FROM Airports;
AVG(ALTITUDE)
826.5714285714286
(1 row)

SELECT MIN(latitude), MAX(altitude) FROM Airports;
MIN(LATITUDE) | MAX(ALTITUDE)
-23.44        | 2459
(1 row)
```

### Le piège : « le **nom** de l'aéroport le plus au sud »

L'aéroport le plus au sud est celui qui a la **plus petite latitude**. On a envie d'écrire `SELECT name, MIN(latitude) FROM Airports;`… mais H2 **refuse** :

*Dans la console H2 :*

```sql
SELECT name, MIN(latitude) FROM Airports;
```

*Résultat dans la console H2 :*

```text
Erreur : La colonne "NAME" doit être dans la liste du GROUP BY
Column "NAME" must be in the GROUP BY list; SQL statement:
SELECT name, MIN(latitude) FROM Airports [90016-252]
```

**Pourquoi ?** `MIN(latitude)` résume les 7 lignes en **une** valeur, alors que `name` a **7** valeurs (une par ligne). H2 ne sait pas comment mettre 7 noms à côté d'un seul nombre. (Le message parle de `GROUP BY`, une notion hors programme : il suffit de retenir que cette requête est refusée.)

**Méthode 1 : trier.** On trie par latitude croissante : le plus au sud arrive en **première ligne**.

*Dans la console H2 :*

```sql
SELECT name, latitude FROM Airports ORDER BY latitude;
```

*Résultat dans la console H2 :*

```text
NAME                      | LATITUDE
Guarulhos Airport         | -23.44
Tokyo Haneda Airport      | 35.55
Madrid-Barajas Airport    | 40.47
Toulouse-Blagnac Airport  | 43.63
Paris-Orly Airport        | 48.72
Paris Issy Heliport       | 48.83
Charles de Gaulle Airport | 49.01
(7 rows)
```

**Méthode 2 : une requête dans la requête** (appelée *sous-requête*). On le fait en deux temps, comme on le ferait à la main :
1. d'abord, quelle est la plus petite latitude ? → `SELECT MIN(latitude) FROM Airports` donne `-23.44` ;
2. ensuite, quel aéroport a cette latitude ? → `SELECT name FROM Airports WHERE latitude = -23.44`.

En SQL, on **met la requête 1 à la place du nombre**, entre parenthèses. H2 calcule d'abord la parenthèse, puis la requête principale :

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE latitude = (SELECT MIN(latitude) FROM Airports);
```

*Résultat dans la console H2 :*

```text
NAME
Guarulhos Airport
(1 row)
```

L'avantage de la méthode 2 : le résultat est **directement** la réponse (une seule ligne), en **une seule requête**.

> Le programme précise qu'on n'utilise **pas** `GROUP BY` (qui ferait un calcul « pour chaque pays » ou « pour chaque continent »). Si on a besoin d'un résultat par pays, on fait une requête par pays.

✏️ **Vérifie que tu as compris :** quelle requête donne le nombre d'héliports ? *(Réponse : `SELECT COUNT(*) FROM Airports WHERE type = 'heliport';` → 1.)*

---
# 9. La clé primaire : reconnaître chaque ligne

**Le problème.** Dans notre table `Countries` de la partie 5, rien n'empêche d'insérer **deux fois** la France. On aurait alors deux lignes identiques : impossible de savoir laquelle modifier ou supprimer. Il faut un moyen de **reconnaître chaque ligne sans ambiguïté**, comme un numéro de carte d'identité.

**Définition (polycopié).** Un **identifiant** (ou **clé**) d'une relation est un attribut, ou un groupe d'attributs, qui permet d'identifier **de manière unique** un enregistrement : deux lignes ne peuvent pas avoir le même identifiant. Il y a parfois plusieurs choix possibles (les **clés candidates**) ; celui qu'on choisit s'appelle la **clé primaire**.

**Comment savoir si un attribut peut être une clé ?** On se pose **une seule question** : *deux objets différents peuvent-ils avoir la même valeur ?*

| attribut des aéroports | deux aéroports peuvent-ils avoir la même valeur ? | clé possible ? |
|:---|:---|:---|
| `city` | oui : trois aéroports à Paris | ✘ |
| `country` | oui : quatre aéroports en France | ✘ |
| `name` | c'est rare, mais rien ne l'interdit | ✘ (pas garanti) |
| `code` | non : chaque aéroport a un code ICAO unique, attribué exprès | ✔ |

C'est pour cela qu'on **invente** souvent un numéro qui sert de clé : numéro d'identifiant `id`, matricule d'un élève, numéro de commande… Dans un schéma, la clé primaire est **soulignée** : Airports(<u>code</u>, name, type, latitude, altitude, country, city).

**Le SGBD fait respecter la clé.** On la déclare avec `PRIMARY KEY (colonne)` dans le `CREATE TABLE`. Ensuite, H2 **refuse** toute ligne qui aurait la même clé qu'une ligne existante (c'est la *contrainte de relation*). Exemple avec une petite table `Pays` dont la clé est le code :

*Dans la console H2 :*

```sql
CREATE TABLE Pays (
    code CHAR(2), name VARCHAR,
    PRIMARY KEY (code) );
INSERT INTO Pays VALUES ('FR', 'France');
INSERT INTO Pays VALUES ('FR', 'France');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Pays ( code CHAR(2), name VARCHAR, PRIMARY KEY (code) );
(Update count: 0)

INSERT INTO Pays VALUES ('FR', 'France');
(Update count: 1)

INSERT INTO Pays VALUES ('FR', 'France');
Erreur : Violation d'index unique ou clé primaire: "PUBLIC.CONSTRAINT_2 INDEX PUBLIC.PRIMARY_KEY_2 ON PUBLIC.PAYS(CODE) VALUES ( /* 1 */ CAST('FR' AS CHAR(2)) )"
Unique index or primary key violation: "PUBLIC.CONSTRAINT_2 INDEX PUBLIC.PRIMARY_KEY_2 ON PUBLIC.PAYS(CODE) VALUES ( /* 1 */ CAST('FR' AS CHAR(2)) )"; SQL statement:
INSERT INTO Pays VALUES ('FR', 'France') [23505-252]
```

- La première insertion est acceptée, la deuxième **refusée** (« violation d'index unique ou clé primaire ») : la France n'existe qu'une fois.
- Une clé peut être faite de **plusieurs** attributs : on les met tous dans la même parenthèse, par exemple `PRIMARY KEY (nom, prenom, date_naissance)`. C'est alors le **groupe** qui doit être unique.

---
# 10. Plusieurs tables : la clé étrangère et la jointure

## 10.1 La clé étrangère : une colonne qui renvoie vers une autre table

Dans la table `Airports`, la colonne `country` contient `'FR'`, `'ES'`, `'BR'`… Ce ne sont pas des noms de pays : ce sont des **codes**, qui **renvoient** vers la table des pays. Pour savoir que `'BR'` veut dire *Brazil*, il faut aller voir dans `Countries`.

| Airports (extrait) | | | → | Countries | |
|:---|:---|:---|:---|:---|:---|
| **code** | **name** | **country** | | **code** | **name** |
| LFPG | Charles de Gaulle Airport | FR | → | FR | France |
| LEMD | Madrid-Barajas Airport | ES | → | ES | Spain |
| SBGR | Guarulhos Airport | BR | → | BR | Brazil |

**Définition (polycopié).** Un attribut qui fait référence à la **clé primaire** d'une autre relation est une **clé étrangère**. Ici, `Airports.country` est une clé étrangère qui fait référence à `Countries.code`. Dans un schéma, on la marque d'un astérisque : Airports(<u>code</u>, name, …, country\*, city).

## 10.2 La jointure : recoller deux tables

**Le besoin :** « Je veux, pour chaque aéroport, son nom **et le nom de son pays**. » Problème : le nom de l'aéroport est dans `Airports`, le nom du pays est dans `Countries`. Aucune des deux tables ne suffit seule.

**L'idée :** la **jointure** fabrique un grand tableau en **recollant** chaque ligne de `Airports` avec **la** ligne de `Countries` qui lui correspond, c'est-à-dire celle qui a le même code de pays. C'est exactement ce que font les flèches du tableau ci-dessus. Pour Charles de Gaulle (`country` = FR), H2 cherche dans `Countries` la ligne où `code` = FR, et colle les deux :

| Airports.code | Airports.name | Airports.country | Countries.code | Countries.name |
|:---|:---|:---|:---|:---|
| LFPG | Charles de Gaulle Airport | FR | FR | France |
| … | … | … | … | … |

Dans ce grand tableau, on a **à la fois** le nom de l'aéroport et le nom du pays : il ne reste plus qu'à choisir les colonnes.

**La syntaxe :**
```sql
SELECT Airports.name, Countries.name
FROM Airports
JOIN Countries ON Airports.country = Countries.code;
```

| morceau | sens |
|:---|:---|
| `FROM Airports` | on part de la table des aéroports |
| `JOIN Countries` | on lui recolle la table des pays |
| `ON Airports.country = Countries.code` | **la règle pour recoller** : la clé étrangère doit être égale à la clé primaire qu'elle référence |
| `SELECT Airports.name, Countries.name` | on affiche le nom de l'aéroport et le nom du pays |

**Pourquoi `Airports.name` et pas juste `name` ?** Les deux tables ont une colonne `name`. Pour dire laquelle on veut, on écrit `table.colonne`. Sans préfixe, H2 répond que la colonne est **ambiguë**.

*Dans la console H2 :*

```sql
SELECT Airports.name, Countries.name
FROM Airports
JOIN Countries ON Airports.country = Countries.code;
```

*Résultat dans la console H2 :*

```text
NAME                      | NAME
Charles de Gaulle Airport | France
Paris-Orly Airport        | France
Toulouse-Blagnac Airport  | France
Paris Issy Heliport       | France
Madrid-Barajas Airport    | Spain
Guarulhos Airport         | Brazil
Tokyo Haneda Airport      | Japan
(7 rows)
```

**Après la jointure, tout le reste marche comme avec une seule table** : `WHERE`, `ORDER BY`, `COUNT`… Le `WHERE` peut utiliser les colonnes des **deux** tables. *« Les aéroports d'Espagne, en ne connaissant que le nom du pays (Spain) »* :

*Dans la console H2 :*

```sql
SELECT Airports.name
FROM Airports
JOIN Countries ON Airports.country = Countries.code
WHERE Countries.name = 'Spain';
```

*Résultat dans la console H2 :*

```text
NAME
Madrid-Barajas Airport
(1 row)
```

✏️ **Vérifie que tu as compris :** pourquoi ne peut-on pas répondre à « les aéroports d'Espagne » avec la seule table `Airports`, en écrivant `WHERE country = 'Spain'` ? *(Réponse : la colonne `country` contient le code `'ES'`, pas le nom `'Spain'` ; le nom n'est que dans `Countries`. Soit on connaît le code et on écrit `country = 'ES'`, soit on fait la jointure.)*

## 10.3 Le SGBD vérifie les clés étrangères : la contrainte de référence

Si on **déclare** la clé étrangère dans le `CREATE TABLE` avec `FOREIGN KEY (colonne) REFERENCES AutreTable(colonne)`, le SGBD vérifie que chaque valeur **existe** dans l'autre table. Exemple : une table `Villes`, dont la colonne `pays` fait référence à la table `Pays` de la partie 9 (qui ne contient que la France).

*Dans la console H2 :*

```sql
CREATE TABLE Villes (
    nom VARCHAR, pays CHAR(2),
    PRIMARY KEY (nom),
    FOREIGN KEY (pays) REFERENCES Pays(code) );
INSERT INTO Villes VALUES ('Lyon', 'FR');
INSERT INTO Villes VALUES ('Rome', 'IT');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Villes ( nom VARCHAR, pays CHAR(2), PRIMARY KEY (nom), FOREIGN KEY (pays) REFERENCES Pays(code) );
(Update count: 0)

INSERT INTO Villes VALUES ('Lyon', 'FR');
(Update count: 1)

INSERT INTO Villes VALUES ('Rome', 'IT');
Erreur : Intégrité référentielle violation de contrainte: "CONSTRAINT_96: PUBLIC.VILLES FOREIGN KEY(PAYS) REFERENCES PUBLIC.PAYS(CODE) (CAST('IT' AS CHAR(2)))"
Referential integrity constraint violation: "CONSTRAINT_96: PUBLIC.VILLES FOREIGN KEY(PAYS) REFERENCES PUBLIC.PAYS(CODE) (CAST('IT' AS CHAR(2)))"; SQL statement:
INSERT INTO Villes VALUES ('Rome', 'IT') [23506-252]
```

- Lyon est acceptée : le pays `FR` existe dans `Pays`.
- Rome est **refusée** : le pays `IT` n'existe pas dans `Pays`. Sans cette vérification, on aurait une ville dans un pays « fantôme ».

**Conséquence sur l'ordre des opérations :**
- pour **créer et remplir** : d'abord la table **référencée** (`Pays`), ensuite celle qui y fait référence (`Villes`) ;
- pour **supprimer** : l'inverse. On ne peut pas supprimer la France tant que Lyon y fait référence.

**Les trois contraintes d'intégrité** (programme officiel) — ce sont les trois vérifications automatiques du SGBD rencontrées dans ce cours :

| contrainte | ce qui est vérifié | exemple refusé (vu dans ce cours) |
|:---|:---|:---|
| de **domaine** | chaque valeur a le bon type | `' IM '` (4 caractères) dans un `CHAR(2)` — partie 5 |
| de **relation** | pas deux lignes avec la même clé primaire | la France insérée deux fois — partie 9 |
| de **référence** | la valeur d'une clé étrangère existe dans l'autre table | une ville dans le pays `IT` inexistant — partie 10 |

---
# 11. Pourquoi plusieurs tables ? (les anomalies)

On pourrait se demander : pourquoi s'embêter avec deux tables et une jointure ? Pourquoi ne pas mettre le nom du pays directement dans la table des aéroports ?

**Essayons :** une seule table, avec le nom du pays répété sur chaque ligne.

| code | name | country_name |
|:---|:---|:---|
| LFPG | Charles de Gaulle Airport | France |
| LFPO | Paris-Orly Airport | France |
| LFBO | Toulouse-Blagnac Airport | France |
| LFPI | Paris Issy Heliport | France |

« France » est écrit **4 fois** : c'est une **redondance** (une information répétée). Avec les milliers d'aéroports français d'une vraie base, il serait écrit des milliers de fois. Cela provoque trois sortes de problèmes, les **anomalies** :
- **anomalie de mise à jour** : si on veut corriger l'orthographe d'un pays, il faut modifier **toutes** ses lignes ; si on en oublie une, la base contient deux orthographes différentes pour le même pays ;
- **anomalie d'insertion** : on ne peut pas enregistrer un nouveau pays tant qu'il n'a aucun aéroport (il n'y a pas de ligne où l'écrire) ;
- **anomalie de suppression** : si on supprime le dernier aéroport d'un pays, on perd aussi le nom de ce pays.

**La solution :** découper en deux tables. Le nom de chaque pays est écrit **une seule fois**, dans `Countries` ; les aéroports ne contiennent que le **code** du pays (la clé étrangère). Quand on a besoin des deux informations ensemble, on les **recolle** avec une jointure. C'est le principe des bases de données **relationnelles**.

---
# 12. Modifier et supprimer des lignes : `UPDATE`, `DELETE`

Les requêtes `SELECT` ne font que lire. Pour **changer** le contenu d'une table, il y a trois requêtes : `INSERT` (ajouter une ligne, partie 5), `UPDATE` (modifier) et `DELETE` (supprimer).

### `UPDATE` : modifier des valeurs

**Le besoin :** « L'altitude d'Orly était fausse : c'est 300, pas 291. »

**La syntaxe :**
```sql
UPDATE Airports SET altitude = 300 WHERE code = 'LFPO';
```

| morceau | sens |
|:---|:---|
| `UPDATE Airports` | on modifie la table `Airports` |
| `SET altitude = 300` | la colonne `altitude` prend la valeur 300 |
| `WHERE code = 'LFPO'` | **seulement** sur les lignes où le code est LFPO |

Le `WHERE` fonctionne exactement comme dans un `SELECT` : il choisit les lignes concernées. On utilise de préférence la **clé primaire**, pour être sûr de ne toucher qu'**une** ligne.

*Dans la console H2 :*

```sql
UPDATE Airports SET altitude = 300 WHERE code = 'LFPO';
SELECT code, altitude FROM Airports WHERE city = 'Paris';
```

*Résultat dans la console H2 :*

```text
UPDATE Airports SET altitude = 300 WHERE code = 'LFPO';
(Update count: 1)

SELECT code, altitude FROM Airports WHERE city = 'Paris';
CODE | ALTITUDE
LFPG | 392
LFPO | 300
LFPI | 112
(3 rows)
```

`Update count: 1` : une ligne a été modifiée. Le `SELECT` qui suit le confirme.

### `DELETE` : supprimer des lignes

**Le besoin :** « On ne garde que les aéroports, on supprime les héliports. »
```sql
DELETE FROM Airports WHERE type = 'heliport';
```
Toutes les lignes qui vérifient la condition sont supprimées (ici une seule).

*Dans la console H2 :*

```sql
DELETE FROM Airports WHERE type = 'heliport';
SELECT COUNT(*) FROM Airports;
```

*Résultat dans la console H2 :*

```text
DELETE FROM Airports WHERE type = 'heliport';
(Update count: 1)

SELECT COUNT(*) FROM Airports;
COUNT(*)
6
(1 row)
```

⚠️ **Le piège le plus grave de tout le chapitre : oublier le `WHERE`.**
- `UPDATE Airports SET altitude = 300;` met l'altitude de **tous** les aéroports à 300 ;
- `DELETE FROM Airports;` supprime **tous** les aéroports.

Il n'y a **pas** de bouton « annuler ». Réflexe : avant un `UPDATE` ou un `DELETE`, faire d'abord un `SELECT` avec le même `WHERE`, pour voir quelles lignes vont être touchées.

---
# 13. La méthode pour écrire une requête

Devant une question en français, on ne commence pas à écrire du SQL au hasard. On répond à **quatre questions, dans cet ordre**. Exemple : *« Quels sont les noms des grands aéroports de France, par ordre alphabétique ? »*

| étape | question à se poser | dans l'exemple | ce qu'on écrit |
|:---|:---|:---|:---|
| 1 | Qu'est-ce qu'on veut **afficher** ? | les noms | `SELECT name` |
| 2 | Dans quelle(s) **table(s)** sont les informations (à afficher **et** à tester) ? | tout est dans `Airports` | `FROM Airports` (avec `JOIN … ON …` s'il faut deux tables) |
| 3 | Quelles **lignes** garde-t-on ? | celles de France, de type grand aéroport | `WHERE country = 'FR' AND type = 'large_airport'` |
| 4 | Faut-il **compter, trier, enlever les doublons** ? | trier par nom | `ORDER BY name` |

Résultat : `SELECT name FROM Airports WHERE country = 'FR' AND type = 'large_airport' ORDER BY name;`

**Traduire les mots de la question :**

| dans la question | en SQL |
|:---|:---|
| « quels sont les… », « donner les… » | `SELECT` les colonnes demandées |
| « de France », « dont l'altitude dépasse… » | `WHERE` |
| « et », « ou » | `AND`, `OR` |
| « contient le mot… », « commence par… » | `LIKE '%mot%'`, `LIKE 'mot%'` |
| « combien » | `COUNT(*)` |
| « la moyenne », « le total » | `AVG`, `SUM` |
| « le plus grand », « le plus petit » | `MAX`, `MIN` (ou `ORDER BY`) ; « le **nom** du plus… » → sous-requête |
| « par ordre alphabétique », « classés par… » | `ORDER BY` (`DESC` pour décroissant) |
| « les différents… », « sans répétition » | `DISTINCT` |
| l'information est dans deux tables | `JOIN … ON clé_étrangère = clé_primaire` |

**Conseils pratiques :**
- commencer par `SELECT * FROM table;` pour voir comment les valeurs sont écrites (`'heliport'` ? `'FR'` ou `'France'` ?) ;
- construire la requête **petit à petit** : d'abord sans `WHERE`, puis ajouter le `WHERE`, puis le tri ; vérifier le résultat à chaque étape.

## Fiche récapitulative

```sql
-- créer, remplir
CREATE TABLE T (a INT, b VARCHAR, PRIMARY KEY (a), FOREIGN KEY (b) REFERENCES U(c));
INSERT INTO T (a, b) VALUES (1, 'texte');
RUNSCRIPT FROM 'chemin/vers/fichier.sql';

-- interroger (ordre d'écriture imposé)
SELECT a, b FROM T WHERE a > 3 AND b LIKE '%mot%' ORDER BY a DESC;
SELECT DISTINCT b FROM T;
SELECT COUNT(*), SUM(a), AVG(a), MIN(a), MAX(a) FROM T WHERE ...;
SELECT b FROM T WHERE a = (SELECT MAX(a) FROM T);

-- plusieurs tables
SELECT T.a, U.d FROM T JOIN U ON T.b = U.c WHERE ...;

-- modifier, supprimer (toujours avec WHERE !)
UPDATE T SET b = 'nouveau' WHERE a = 1;
DELETE FROM T WHERE a = 1;
```

| mot | sens |
|:---|:---|
| relation / attribut / enregistrement | table / colonne / ligne |
| domaine | type des valeurs d'un attribut |
| schéma | liste des attributs et de leurs types, clé primaire soulignée, clés étrangères marquées |
| clé primaire | identifie chaque ligne de façon unique |
| clé étrangère | fait référence à la clé primaire d'une autre table |
| jointure | recolle deux tables grâce à une clé étrangère |
| SGBD | logiciel qui gère la base : persistance, accès concurrents, efficacité, sécurité |

---
# 14. *(Pour le professeur)* Quelle partie pour quel exercice du polycopié

| exercice du polycopié | parties du cours à avoir lues |
|:---|:---|
| Ex. 1 : création d'une table, import des données | 1 à 6 |
| Ex. 2 : aéroports | 7, 8 et 10 (jointures à partir de la question 3) |
| Ex. 3 : Vélib' | 7, 8 et 10 |
| Ex. 4 et 5 : clés primaires | 9 |
| Ex. 6 : achats, clés étrangères | 9 et 10 |
| Ex. 7 : location de voitures | toutes, dont 12 |

> ⚠️ **Liberté prise :** les données des exemples (quelques pays et aéroports) sont une petite sélection faite pour ce cours ; les vraies tables des exercices sont beaucoup plus grandes et ont d'autres identifiants. Les résultats ont été obtenus avec H2 et recopiés (sans la durée d'exécution, et avec « Erreur : » à la place du nom technique de l'erreur Java).